#### Search online and figure out what gating means in a recommendation system. Implement a simple gating in PyTorch. Reason about the difference between gating and attention.

In [4]:
import numpy as np
import math

import torch
import torch.nn as nn
import torch.nn.functional as F
from typing import Tuple

In [ ]:
# ============================================================
# SIMPLE GATING: Mixture of Experts for Recommendation
# ============================================================

class Expert(nn.Module):
    """Single expert network (e.g., for a specific domain or task)."""
    def __init__(self, input_dim: int, hidden_dim: int, output_dim: int):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, output_dim)
        )
    
    def forward(self, x):
        return self.net(x)


class GatingNetwork(nn.Module):
    """Gate that learns to route input to different experts or weight them."""
    def __init__(self, input_dim: int, n_experts: int):
        super().__init__()
        self.router = nn.Linear(input_dim, n_experts)
    
    def forward(self, x):
        """Return gating weights (normalized)."""
        logits = self.router(x)  # (batch, n_experts)
        weights = F.softmax(logits, dim=-1)  # (batch, n_experts) -> sum to 1
        return weights


class MixtureOfExperts(nn.Module):
    """Mixture of Experts layer with gating."""
    def __init__(self, input_dim: int, hidden_dim: int, output_dim: int, n_experts: int = 3):
        super().__init__()
        self.experts = nn.ModuleList([
            Expert(input_dim, hidden_dim, output_dim)
            for _ in range(n_experts)
        ])
        self.gate = GatingNetwork(input_dim, n_experts)
    
    def forward(self, x):
        """Route input through experts and aggregate by gating weights."""
        # Get gating weights
        gate_weights = self.gate(x)  # (batch, n_experts)
        
        # Compute all expert outputs
        expert_outputs = torch.stack([
            expert(x) for expert in self.experts
        ], dim=1)  # (batch, n_experts, output_dim)
        
        # Weight experts by gating weights
        output = torch.einsum('be,beo->bo', gate_weights, expert_outputs)
        # or: output = (gate_weights.unsqueeze(2) * expert_outputs).sum(dim=1)
        
        return output, gate_weights

In [ ]:
# Corrected SimpleAttention (multi-head, shape-safe)

class SimpleAttention(nn.Module):
    """Multi-head attention (single layer) that supports queries of shape (B, D) or (B, Q, D)
    over a context of shape (B, L, D). Returns attention averaged over heads.
    """
    def __init__(self, dim: int, n_heads: int = 4):
        super().__init__()
        assert dim % n_heads == 0, "dim must be divisible by n_heads"
        self.n_heads = n_heads
        self.dim = dim
        self.head_dim = dim // n_heads

        self.query = nn.Linear(dim, dim)
        self.key = nn.Linear(dim, dim)
        self.value = nn.Linear(dim, dim)
        self.output = nn.Linear(dim, dim)

    def _split_heads(self, x: torch.Tensor) -> torch.Tensor:
        """(B, S, D) -> (B, H, S, head_dim)"""
        B, S, D = x.shape
        x = x.view(B, S, self.n_heads, self.head_dim).transpose(1, 2)
        return x  # (B, H, S, head_dim)

    def _merge_heads(self, x: torch.Tensor) -> torch.Tensor:
        """(B, H, S, head_dim) -> (B, S, D)"""
        B, H, S, Hd = x.shape
        x = x.transpose(1, 2).contiguous().view(B, S, H * Hd)
        return x  # (B, S, D)

    def forward(self, x, context):
        """
        x: (B, D) or (B, Q, D)  -> queries
        context: (B, L, D)       -> keys/values over sequence length L
        Returns:
          output: (B, D) if single query, else (B, Q, D)
          attn_weights: mean over heads -> (B, L) if single query, else (B, Q, L)
        """
        # Project
        Q = self.query(x)          # (B, D) or (B, Q, D)
        K = self.key(context)      # (B, L, D)
        V = self.value(context)    # (B, L, D)

        single_query = Q.dim() == 2
        if single_query:
            Q = Q.unsqueeze(1)     # (B, 1, D)

        # Shapes to heads
        B, Qlen, _ = Q.shape
        Bc, L, _ = K.shape
        assert B == Bc, "Batch mismatch between query and context"

        Qh = self._split_heads(Q)  # (B, H, Q, Hd)
        Kh = self._split_heads(K)  # (B, H, L, Hd)
        Vh = self._split_heads(V)  # (B, H, L, Hd)

        # Scaled dot-product attention per head
        scores = torch.matmul(Qh, Kh.transpose(-2, -1))  # (B, H, Q, L)
        scores = scores / math.sqrt(self.head_dim) # Logits' variance scales with Hd, making softmax increasingly peaky as Hd grows, which saturates and yields small/unstable gradients.
        attn = F.softmax(scores, dim=-1)                 # (B, H, Q, L)

        # Weighted sum of values
        out_heads = torch.matmul(attn, Vh)               # (B, H, Q, Hd)
        out = self._merge_heads(out_heads)               # (B, Q, D)
        out = self.output(out)                           # (B, Q, D)

        # Average attention weights over heads for readability
        attn_mean = attn.mean(dim=1)                     # (B, Q, L)

        if single_query:
            out = out.squeeze(1)                         # (B, D)
            attn_mean = attn_mean.squeeze(1)            # (B, L)

        return out, attn_mean

In [32]:
# ============================================================
# DEMO: Gating vs Attention in a Recommendation Scenario
# ============================================================

batch_size = 32
input_dim = 16  # user/item feature dim
hidden_dim = 32
output_dim = 8  # output embedding dim
n_items = 100

print("="*60)
print("GATING vs ATTENTION in Recommendation Systems")
print("="*60)

# Create a batch of user features
user_features = torch.randn(batch_size, input_dim)
item_features = torch.randn(n_items, input_dim)

# --- Gating (Mixture of Experts) ---
print("\n1. GATING (Mixture of Experts)")
print("-" * 60)
moe = MixtureOfExperts(input_dim, hidden_dim, output_dim, n_experts=3)
user_embeddings_gated, gate_weights = moe(user_features)

print(f"User embeddings shape: {user_embeddings_gated.shape}")
print(f"Gating weights shape: {gate_weights.shape}")
print(f"First user gating weights (sum={gate_weights[0].sum():.3f}): {gate_weights[0].detach().numpy()}")
print(f"Interpretation: Expert 0: {gate_weights[0, 0]:.1%}, Expert 1: {gate_weights[0, 1]:.1%}, Expert 2: {gate_weights[0, 2]:.1%}")

# --- Attention (multi-head) ---
print("\n2. ATTENTION (Multi-head)")
print("-" * 60)
attn = SimpleAttention(dim=input_dim, n_heads=4)

# Attend over item features
user_embeddings_attn, attn_weights = attn(user_features, item_features.unsqueeze(0).expand(batch_size, -1, -1))

print(f"User embeddings shape: {user_embeddings_attn.shape}")
print(f"Attention weights shape: {attn_weights.shape}")
print(f"First user attention weights over first 5 items (sum={attn_weights[0, :5].sum():.3f}): {attn_weights[0, :5].detach().numpy()}")
print(f"Interpretation: Dense weights distributed over {n_items} items (soft attention)")

# --- Key Differences ---
print("\n3. KEY DIFFERENCES")
print("-" * 60)
print(f"Gating selects between {len(moe.experts)} experts (sparse routing)")
print(f"Attention distributes weights over {n_items} items (dense aggregation)")
print()
print(f"Gating weights sum to 1 over COMPONENTS: {gate_weights[0].sum():.3f}")
print(f"Attention weights sum to 1 over SEQUENCE: {attn_weights[0].sum():.3f}")
print()
print("Gating is MORE INTERPRETABLE: 'Use expert A for this user'")
print("Attention is MORE FLEXIBLE: 'Combine information from all items'")
print()
print("In recommendation systems:")
print("  - Use GATING for: multi-task learning, domain-specific experts, efficient routing")
print("  - Use ATTENTION for: capturing user interest patterns, temporal dynamics, cross-feature interactions")

GATING vs ATTENTION in Recommendation Systems

1. GATING (Mixture of Experts)
------------------------------------------------------------
User embeddings shape: torch.Size([32, 8])
Gating weights shape: torch.Size([32, 3])
First user gating weights (sum=1.000): [0.46219847 0.26192936 0.27587226]
Interpretation: Expert 0: 46.2%, Expert 1: 26.2%, Expert 2: 27.6%

2. ATTENTION (Multi-head)
------------------------------------------------------------
User embeddings shape: torch.Size([32, 16])
Attention weights shape: torch.Size([32, 100])
First user attention weights over first 5 items (sum=0.048): [0.01110113 0.00886623 0.00997495 0.00926261 0.00843928]
Interpretation: Dense weights distributed over 100 items (soft attention)

3. KEY DIFFERENCES
------------------------------------------------------------
Gating selects between 3 experts (sparse routing)
Attention distributes weights over 100 items (dense aggregation)

Gating weights sum to 1 over COMPONENTS: 1.000
Attention weights sum

#### Implement a pairwise BPR loss calculation for a trio (user, item_i, item_j) where item_i is an item the user interacted with and item_j is one they did not.

In [38]:
# ============================================================
# Bayesian Personalized Ranking (BPR) Loss
# ============================================================

"""
BPR Loss Formula:
    L_BPR = -ln σ(x_uij)
    
where:
    x_uij = ŷ_ui - ŷ_uj  (difference in predicted scores)
    σ(x) = 1 / (1 + exp(-x))  (sigmoid function)
    
Or equivalently:
    L_BPR = -ln σ(ŷ_ui - ŷ_uj)
          = ln(1 + exp(-(ŷ_ui - ŷ_uj)))
          = softplus(ŷ_uj - ŷ_ui)

Interpretation:
- Maximizes the probability that item_i (positive) ranks higher than item_j (negative)
- The sigmoid models P(item_i >_u item_j) = σ(ŷ_ui - ŷ_uj)
- Loss is minimized when ŷ_ui >> ŷ_uj (positive item scores much higher)
"""

def bpr_loss(user_emb, item_i_emb, item_j_emb):
    """Compute BPR loss for a batch of (user, pos_item, neg_item) triplets.
    
    Args:
        user_emb: (batch, dim) - user embeddings
        item_i_emb: (batch, dim) - positive item embeddings (interacted)
        item_j_emb: (batch, dim) - negative item embeddings (not interacted)
    
    Returns:
        loss: scalar tensor (mean BPR loss over batch)
    """
    # Compute predicted scores (dot product)
    score_i = torch.sum(user_emb * item_i_emb, dim=-1)  # (batch,)
    score_j = torch.sum(user_emb * item_j_emb, dim=-1)  # (batch,)
    
    # BPR loss: -ln σ(score_i - score_j) = softplus(score_j - score_i)
    # Using log-sigmoid for numerical stability
    loss = -F.logsigmoid(score_i - score_j).mean()
    
    return loss


# Alternative implementation using softplus (equivalent)
def bpr_loss_softplus(user_emb, item_i_emb, item_j_emb):
    """BPR loss using softplus formulation."""
    score_i = torch.sum(user_emb * item_i_emb, dim=-1)
    score_j = torch.sum(user_emb * item_j_emb, dim=-1)
    
    # softplus(x) = ln(1 + exp(x))
    loss = F.softplus(score_j - score_i).mean()
    
    return loss


# ============================================================
# Demo: BPR Loss Computation
# ============================================================

# Create synthetic embeddings
batch_size = 32
dim = 64

user_emb = torch.randn(batch_size, dim)
item_i_emb = torch.randn(batch_size, dim)  # positive items
item_j_emb = torch.randn(batch_size, dim)  # negative items

# Compute loss
loss = bpr_loss(user_emb, item_i_emb, item_j_emb)
loss_alt = bpr_loss_softplus(user_emb, item_i_emb, item_j_emb)

print("BPR Loss Demo")
print("=" * 60)
print(f"Batch size: {batch_size}")
print(f"Embedding dim: {dim}")
print(f"\nBPR loss (logsigmoid): {loss:.4f}")
print(f"BPR loss (softplus):   {loss_alt:.4f}")

# Verify they're equivalent
print(f"\nDifference: {abs(loss - loss_alt):.6f} (should be ~0)")

# Show score distributions
with torch.no_grad():
    score_i = torch.sum(user_emb * item_i_emb, dim=-1)
    score_j = torch.sum(user_emb * item_j_emb, dim=-1)
    
    print(f"\nScore statistics:")
    print(f"  Positive items (i): mean={score_i.mean():.3f}, std={score_i.std():.3f}")
    print(f"  Negative items (j): mean={score_j.mean():.3f}, std={score_j.std():.3f}")
    print(f"  Score difference (i-j): mean={(score_i - score_j).mean():.3f}")
    print(f"  Fraction where i > j: {(score_i > score_j).float().mean():.1%}")
    
print("\nNote: With random embeddings, scores are ~equal. After training,")
print("      positive items should score higher → loss decreases.")

BPR Loss Demo
Batch size: 32
Embedding dim: 64

BPR loss (logsigmoid): 4.1611
BPR loss (softplus):   4.1611

Difference: 0.000000 (should be ~0)

Score statistics:
  Positive items (i): mean=0.758, std=8.192
  Negative items (j): mean=0.054, std=7.146
  Score difference (i-j): mean=0.704
  Fraction where i > j: 56.2%

Note: With random embeddings, scores are ~equal. After training,
      positive items should score higher → loss decreases.


#### Design a simple feature embedding + MLP architecture for a ranking model that has the following inputs: user ID, item ID, and one continuous feature (e.g., time of day). Outline the model: include embedding lookups, concatenation of features, a couple of dense layers, and an output layer that predicts a score.

# Ranking Model: Embedding + MLP Architecture

This section implements a simple pointwise + pairwise ranking model that combines:

- User & Item ID embeddings
- Optional cyclical time-of-day encoding (sin/cos) projected to an embedding
- Concatenation of feature embeddings -> multi-layer perceptron (MLP) -> score
- Two training objectives demonstrated: BCE (pointwise) and BPR (pairwise)
- Basic evaluation metrics: NDCG@K and item coverage

Workflow:
1. Define `RankingModel`
2. Helper functions: cyclical time encoding, NDCG@K, coverage
3. Synthetic data generation (user-item interactions with implicit feedback)
4. Pointwise BCE training phase
5. Pairwise BPR fine-tuning phase (sample triplets from positives/negatives)
6. Metric evaluation on a held-out mini-batch

Notes:
- Synthetic data is randomly generated for demonstration only.
- In real usage, positives come from implicit signals (click, play, add-to-cart); negatives are sampled.
- Time feature can capture periodic usage patterns (hour-of-day).

# Exploration Logging & Practical Considerations

Events to Log During Exploration/Serving:
- Exposure (user_id, item_id, position, timestamp, variant/control flag)
- Interaction signals (click, play, dwell_time, completion_rate, add_to_cart, purchase)
- Context (hour_of_day, device_type, locale, referrer, session_id)
- Model scores (pre-re-ranking score, post-re-ranking score, exploration probability ε or sampling temperature)
- Re-ranking adjustments (diversity/fairness penalties applied, selected reason codes)
- Negative feedback (skips, hides, dislikes) for counterfactual modeling
- Latency measurements (end-to-end, model inference, feature fetch)
- A/B assignment & bucket versions (for rollback tracking)

Practical Considerations:
- ε Scheduling: start higher (e.g., 0.2) decay to ~0.05; or adaptive per-user based on history length.
- Contextual Bandits: replace uniform exploration with reward model using context → better sample efficiency.
- Safety Rails: cap number of low-relevance exploratory items per page; ensure minimum relevance threshold.
- Diversity Pairing: bias exploratory picks toward under-represented categories or new arrivals.
- Logging Consistency: immutable event schema, version fields for model & feature set, late-arriving event handling.
- Offline Evaluation: IPS / DR estimators for counterfactual CTR when using logged propensities (store p(action|context)).
- Bias Mitigation: randomize among equally scored items to reduce position bias accumulation.
- Monitoring: track exploration rate, reward gap (exploitation vs exploration), novelty engagement, coverage, tail performance.
- Rollback Strategy: feature flag for exploration subsystem; graceful degradation to pure exploitation if anomalies detected.

In [ ]:
# --------------------------------------------------------------
# Helper: Cyclical time (hour-of-day) encoding -> (sin, cos)
# --------------------------------------------------------------

def encode_hour_of_day(hours: torch.Tensor) -> torch.Tensor:
    """Convert hour integers [0,23] to 2D cyclical representation.
    Args:
        hours: (batch,) int tensor of hour-of-day
    Returns:
        features: (batch, 2) tensor with sin/cos encoding
    """
    hours = hours.float()
    angle = 2 * math.pi * (hours / 24.0)
    return torch.stack([torch.sin(angle), torch.cos(angle)], dim=-1)

# --------------------------------------------------------------
# Metrics: NDCG@K & Coverage
# --------------------------------------------------------------

def ndcg_at_k(scores: torch.Tensor, labels: torch.Tensor, k: int) -> float:
    """Compute NDCG@K for a single user's scored items.
    Args:
        scores: (N,) predicted scores
        labels: (N,) relevance (0/1 or graded)
        k: cutoff
    Returns:
        ndcg: float
    """
    N = scores.shape[0]
    k = min(k, N)
    # Sort by predicted scores desc
    idx = torch.argsort(scores, descending=True)
    topk_idx = idx[:k]
    gains = labels[topk_idx]
    # DCG
    ranks = torch.arange(1, k + 1, dtype=torch.float32)
    dcg = torch.sum(gains / torch.log2(ranks + 1))
    # IDCG (ideal ordering by true relevance)
    ideal_idx = torch.argsort(labels, descending=True)[:k]
    ideal_gains = labels[ideal_idx]
    idcg = torch.sum(ideal_gains / torch.log2(ranks + 1))
    if idcg.item() == 0:
        return 0.0
    return (dcg / idcg).item()

def item_coverage(recommended_item_ids: torch.Tensor, total_items: int) -> float:
    """Coverage = fraction of unique recommended items over catalog size."""
    unique = torch.unique(recommended_item_ids)
    return (unique.numel() / total_items)

# --------------------------------------------------------------
# Ranking Model
# --------------------------------------------------------------

class RankingModel(nn.Module):
    def __init__(self, num_users: int, num_items: int, emb_dim: int = 32, time_proj_dim: int = 8, hidden_dims=(64, 32)):
        super().__init__()
        self.user_emb = nn.Embedding(num_users, emb_dim)
        self.item_emb = nn.Embedding(num_items, emb_dim)
        # Project cyclical (2D) time features -> time embedding
        self.time_proj = nn.Linear(2, time_proj_dim)
        layers = []
        input_dim = emb_dim * 2 + time_proj_dim
        prev = input_dim
        for h in hidden_dims:
            layers.append(nn.Linear(prev, h))
            layers.append(nn.ReLU())
            prev = h
        layers.append(nn.Linear(prev, 1))  # final score logit
        self.mlp = nn.Sequential(*layers)

    def forward(self, user_ids: torch.Tensor, item_ids: torch.Tensor, hours: torch.Tensor) -> torch.Tensor:
        """Pointwise forward producing logits.
        Args:
            user_ids: (batch,)
            item_ids: (batch,)
            hours: (batch,) hour-of-day [0,23]
        Returns:
            logits: (batch,) unnormalized scores
        """
        u = self.user_emb(user_ids)
        i = self.item_emb(item_ids)
        t_raw = encode_hour_of_day(hours)  # (batch,2)
        t = self.time_proj(t_raw)          # (batch,time_proj_dim)
        x = torch.cat([u, i, t], dim=-1)
        logits = self.mlp(x).squeeze(-1)
        return logits

    def score(self, user_ids: torch.Tensor, item_ids: torch.Tensor, hours: torch.Tensor) -> torch.Tensor:
        """Convenience alias for forward (for clarity in evaluation)."""
        return self.forward(user_ids, item_ids, hours)

# --------------------------------------------------------------
# Pairwise BPR Step (using embeddings directly)
# --------------------------------------------------------------

def bpr_step(model: RankingModel, user_ids: torch.Tensor, pos_item_ids: torch.Tensor, neg_item_ids: torch.Tensor, hours: torch.Tensor) -> torch.Tensor:
    """Compute BPR loss using model's embeddings.
    We reuse the model's internal embeddings; time feature shared for pos/neg for simplicity.
    """
    u = model.user_emb(user_ids)
    i_pos = model.item_emb(pos_item_ids)
    i_neg = model.item_emb(neg_item_ids)
    # Dot product scores
    score_pos = (u * i_pos).sum(dim=-1)
    score_neg = (u * i_neg).sum(dim=-1)
    loss = -F.logsigmoid(score_pos - score_neg).mean()
    return loss

In [ ]:
# Synthetic Training Demo (Pointwise BCE + Pairwise BPR)

device = torch.device('cpu')

# Hyperparameters
num_users = 200
num_items = 500
emb_dim = 32
batch_size = 256
bce_epochs = 3
bpr_epochs = 2
K_eval = 10

model = RankingModel(num_users, num_items, emb_dim=emb_dim).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

bce_loss_fn = nn.BCEWithLogitsLoss()

# --------------------------------------------------------------
# Synthetic implicit feedback generation
# --------------------------------------------------------------
# Create a simple user preference vector; users prefer items whose id modulo a number matches user modulo.
item_preference_factor = torch.randint(low=0, high=20, size=(num_items,))  # group ids for items
user_preference_bias = torch.randint(low=0, high=20, size=(num_users,))    # preferred group per user

# Function to sample a batch

def sample_pointwise_batch(batch_size: int) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor]:
    user_ids = torch.randint(0, num_users, (batch_size,))
    item_ids = torch.randint(0, num_items, (batch_size,))
    hours = torch.randint(0, 24, (batch_size,))
    # Label: 1 if group's match + random noise, else 0
    group_match = (item_preference_factor[item_ids] == user_preference_bias[user_ids]).float()
    # Add stochasticity
    prob = 0.7 * group_match + 0.1 * torch.rand_like(group_match)
    labels = torch.bernoulli(prob.clamp(0, 1))
    return user_ids, item_ids, hours, labels

# --------------------------------------------------------------
# Pointwise BCE Training
# --------------------------------------------------------------
print("Pointwise BCE training...")
for epoch in range(1, bce_epochs + 1):
    user_ids, item_ids, hours, labels = sample_pointwise_batch(batch_size)
    logits = model(user_ids, item_ids, hours)
    loss = bce_loss_fn(logits, labels)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    with torch.no_grad():
        preds = torch.sigmoid(logits)
        avg_score_pos = preds[labels == 1].mean().item() if (labels == 1).any() else float('nan')
        avg_score_neg = preds[labels == 0].mean().item() if (labels == 0).any() else float('nan')
    print(f"Epoch {epoch}: BCE Loss={loss.item():.4f} avg_pos_score={avg_score_pos:.3f} avg_neg_score={avg_score_neg:.3f}")

# --------------------------------------------------------------
# Pairwise BPR Fine-tuning
# --------------------------------------------------------------
print("\nPairwise BPR fine-tuning...")
for epoch in range(1, bpr_epochs + 1):
    # Sample a larger batch and build triplets
    user_ids, item_ids, hours, labels = sample_pointwise_batch(batch_size * 2)
    pos_mask = labels == 1
    neg_mask = labels == 0
    pos_users = user_ids[pos_mask]
    pos_items = item_ids[pos_mask]
    neg_users = user_ids[neg_mask]
    neg_items = item_ids[neg_mask]
    # Align sizes by random sampling
    if pos_users.numel() == 0 or neg_users.numel() == 0:
        continue
    sample_size = min(pos_users.numel(), neg_users.numel(), 256)
    idx_pos = torch.randint(0, pos_users.numel(), (sample_size,))
    idx_neg = torch.randint(0, neg_users.numel(), (sample_size,))
    triplet_users = pos_users[idx_pos]
    pos_item_ids = pos_items[idx_pos]
    neg_item_ids = neg_items[idx_neg]
    triplet_hours = hours[:sample_size]  # reuse some hours

    loss_bpr = bpr_step(model, triplet_users, pos_item_ids, neg_item_ids, triplet_hours)
    optimizer.zero_grad()
    loss_bpr.backward()
    optimizer.step()
    print(f"BPR Epoch {epoch}: Loss={loss_bpr.item():.4f}")

Pointwise BCE training...
Epoch 1: BCE Loss=0.6933 avg_pos_score=0.493 avg_neg_score=0.499
Epoch 2: BCE Loss=0.6735 avg_pos_score=0.484 avg_neg_score=0.488
Epoch 3: BCE Loss=0.6553 avg_pos_score=0.465 avg_neg_score=0.475

Pairwise BPR fine-tuning...
BPR Epoch 1: Loss=3.5292
BPR Epoch 2: Loss=2.7214


In [31]:
# --------------------------------------------------------------
# Evaluation Demo for a Single User (NDCG@K)
# --------------------------------------------------------------
user_eval = torch.randint(0, num_users, (1,)).item()
# Score all items for this user at a fixed hour
eval_hours = torch.full((num_items,), 12)
user_ids_full = torch.full((num_items,), user_eval)
with torch.no_grad():
    scores = model.score(user_ids_full, torch.arange(num_items), eval_hours)
    # Construct pseudo labels using original preference heuristic
    labels_full = (item_preference_factor == user_preference_bias[user_eval]).float()
    ndcg = ndcg_at_k(scores, labels_full, K_eval)
    # Top-K item coverage relative to entire catalog
    topk_items = torch.argsort(scores, descending=True)[:K_eval]
    coverage = item_coverage(item_preference_factor[topk_items], item_preference_factor.unique().numel())

print(f"\nUser {user_eval} NDCG@{K_eval}: {ndcg:.4f}")
print(f"Top-{K_eval} coverage fraction: {coverage:.4f}")
print("Top-K item IDs:", topk_items.tolist())
print("Relevant among Top-K:", labels_full[topk_items].tolist())


User 80 NDCG@10: 0.0000
Top-10 coverage fraction: 0.4500
Top-K item IDs: [391, 121, 145, 334, 478, 173, 392, 172, 2, 107]
Relevant among Top-K: [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]
